In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import logging
import os
from scipy.stats import ks_2samp
import numpy as np
import json

from gsm_benchmarker.results_analyser.prompt_result import PromptResult, MultiPromptResult
from gsm_benchmarker.results_analyser.plotting_utils import Colour


logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
METRIC = "correct"

In [ ]:
ALPHA = 0.05

In [ ]:
OUTPUTS = Path("results_pres/outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
with open('mirzadeh-data.json') as f:
    original_results_df = pd.DataFrame(json.load(f))
    original_model_order = original_results_df.model.tolist()

original_model_order


In [ ]:
result_kwargs = dict(metric=METRIC, save_dest=OUTPUTS)
pp = Path("../../../data/gsm-symbolic/outputs").resolve()


gsm_result = PromptResult(
    pp / "noq_default_full__12_05/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)

nonformal_result = PromptResult(
    pp / "noq_nonformalised__12_05/final",
    colour=Colour("skyblue"),
    full_label="Simple NL prompt",
    short_label="NL-simple",
    baseline=gsm_result.mres,
    **result_kwargs
)

formal_result = PromptResult(
    pp / "noq_formalised__12_05/final",
    colour=Colour("steelblue"),
    full_label="Structured NL prompt",
    short_label="NL-structured",
    baseline=gsm_result.mres,
    **result_kwargs
)

short_code_result = PromptResult(
    pp / "noq_code_short__12_05/final",
    colour=Colour("mediumpurple").lighten(0.2),
    full_label="Simple code-output prompt",
    short_label="code-simple",
    baseline=gsm_result.mres,
    **result_kwargs
)

long_code_result = PromptResult(
    pp / "noq_code_long__12_05/final",
    colour=Colour("rebeccapurple"),
    full_label="Structured code-output prompt",
    short_label="code-structured",
    baseline=gsm_result.mres,
    **result_kwargs
)

full_results = {
    'gsm': gsm_result,
    'NL-simple': nonformal_result,
    'NL-structured': formal_result,
    'code-simple': short_code_result,
    'code-structured': long_code_result
}


In [ ]:
_ = gsm_result.mres.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

### Evaluating number distribution shift in GSM-Variants w.r.t. GSM-Base

In [ ]:
number_counts = gsm_result.mres.get_number_counts()[0]
numeric_index = pd.to_numeric(number_counts.index)

gsm8k_samples = np.repeat(numeric_index, number_counts['GSM-Base'].values)
main_samples = np.repeat(numeric_index, number_counts['GSM-Variants'].values)

ks_stat, p_value = ks_2samp(gsm8k_samples, main_samples)

print(f"K-S Statistic: {ks_stat:.4f}")
print(f"P-value: {p_value:.4e}")

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

In [ ]:
gsm_result.plot_variant_effect()

In [ ]:
gsm_result.variant_effect_to_latex(model_order=original_model_order)

In [ ]:
significant_models = gsm_result.get_significant_models(alpha=ALPHA, drop_only=True)
significant_models

In [ ]:
len(significant_models)

In [ ]:
for res in (nonformal_result, formal_result, short_code_result, long_code_result):
    res.models = significant_models


## Question 2
Do alternative prompt formats remove the variant dependency?

In [ ]:
nonformal_result.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
nonformal_result.variant_effect_to_latex(model_order=significant_models)

In [ ]:
formal_result.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
formal_result.variant_effect_to_latex(model_order=significant_models)

In [ ]:
short_code_result.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
short_code_result.variant_effect_to_latex(model_order=significant_models)

In [ ]:
long_code_result.plot_variant_effect(model_order=significant_models[::-1])

In [ ]:
long_code_result.variant_effect_to_latex(model_order=significant_models)

In [ ]:
# models to be used for plots later - the ones that show significant variant effect on at least one other prompt
demo_models = [
    'gemma-2b',
    'gemma-2-2b',
    'gemma-7b-it',
    'phi-2',
    'Meta-Llama-3-8B',
    'Meta-Llama-3-8B-Instruct',
]


### Summary of all prompts and models


In [ ]:
all_prompts_result = MultiPromptResult(full_results, save_prefix=OUTPUTS_FOLDER)
all_prompts_result.summary

In [ ]:
fig = all_prompts_result.plot_prompt_comparison(models=demo_models, add_bar_labels=True, x_labels_rotation=5)

In [ ]:
fig = all_prompts_result.plot_prompt_acc_evolution(models=demo_models, n_cols=2, sharex=False, sharey=False, equal_aspect=False, figsize=(10, 10), bottom_margin=.06)


### Diagnostics data for the variant effect fits

In [ ]:
def make_fit_diagnostics_summary(name):

    all_diagnostics = []
    for prompt_format, res in full_results.items():  # your dict of {format_name: diagnostics_df}
        df = getattr(res, name)[1].reset_index()
        df['prompt_format'] = prompt_format
        all_diagnostics.append(df)

    combined_diagnostics = pd.concat(all_diagnostics, ignore_index=True)
    total_fits = len(combined_diagnostics)
    clean_fits = combined_diagnostics[
        (~combined_diagnostics['fit_failed']) &
        (~combined_diagnostics['is_singular']) &
        (combined_diagnostics['convergence_messages'] == '')
    ]

    cond = combined_diagnostics['is_singular'] | (combined_diagnostics['convergence_messages'] != '')
    flagged_fits = combined_diagnostics[cond]

    print(f"{len(clean_fits)} / {total_fits} fits converged cleanly")
    print(flagged_fits[['model', 'prompt_format', 'is_singular', 'convergence_messages']])
    print()

    if flagged_fits.size:
        print("Convergence messages:")
        for i in range(flagged_fits.shape[0]):
            print(f"#{flagged_fits.index[i]}", flagged_fits.iloc[i].convergence_messages)
            print()

    print("Clean fits summary")
    print(clean_fits['ranef_variance'].agg(['min', 'max', 'mean']))


In [ ]:
make_fit_diagnostics_summary('variant_effect')

### Number effect tables

In [ ]:
# number effect - GLMM results
all_prompts_result.number_effect_to_latex("number_effect", models=significant_models)

In [ ]:
# number-effect-corrected variant effect - GLMM results
all_prompts_result.number_effect_to_latex("delta_symb_ne", models=significant_models)

In [ ]:
make_fit_diagnostics_summary('number_effect')


In [41]:
long_code_result.number_effect[1]

,fit_failed,is_singular,convergence_messages,ranef_variance,ranef_sd
model,,,,,
Mathstral-7B-v0.1,False,False,,19.067956,4.366687
Meta-Llama-3-8B,False,False,,15.797836,3.974649
Meta-Llama-3-8B-Instruct,False,False,,9.618717,3.101406
Mistral-7B-Instruct-v0.1,False,False,,10.787091,3.284371
Phi-3.5-mini-instruct,False,False,,26.256294,5.124090
gemma-2-2b,False,False,Model failed to converge with max|grad| = 0.16...,7.946997,2.819042
gemma-2-9b,False,False,,16.748927,4.092545
gemma-2b,False,False,,6.274512,2.504898
gemma-7b-it,False,False,,16.217206,4.027059
